# Celda 1: Configuración de Librerías y Entorno
Es fundamental mostrar que usas un entorno virtual y gestionas credenciales de forma segura

In [2]:
import os
import pyodbc
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

# Cargamos variables de entorno (.env) para seguridad
load_dotenv()

True

# Celda 2: Fase 1 - Conexión y Extracción de SQL 
Aquí demuestras tu habilidad para extraer datos binarios (BLOBs) de la base de datos de la Fiscalía

In [6]:
# Fase 1: Conexión y Extracción de SQL Server (Carga de archivos .doc)
import pyodbc
import os

# Configuración de conexión local
conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=JOEL-LT;"
    "DATABASE=NDFPCYFCH;"
    "UID=CPFSQL2025;"
    "PWD=A1JOEL*@#;"
)

def extraer_documentos_fiscalia():
    # Definimos la ruta de salida (relativa a la carpeta del notebook)
    output_dir = '../documentos'
    if not os.path.exists(output_dir): 
        os.makedirs(output_dir)
    
    # Consulta SQL para extraer los 19 registros específicos
    query = """
    SELECT TOP 19
        L.[DOCUMENTO], 
        L.[EXT_DOC_CASO], 
        N.[DESPACHO],
        N.[COD_NUM_DOC],
        C.[AÑOCASO],
        C.[NUMCASO],
        C.[SECCUENCIACASO],
        N.[TIPO_DOCUMENTO]
    FROM [NDFPCYFCH].[dbo].[LEGADOCCASO] L
    INNER JOIN [NDFPCYFCH].[dbo].[NUMERODOC] N ON L.DESPACHO = N.DESPACHO AND L.COD_NUM_DOC = N.COD_NUM_DOC
    INNER JOIN [NDFPCYFCH].[dbo].[CASO] C ON N.DESPACHO = C.CODDESPACHO AND N.COD_NUM_DOC = C.CODNUMDOC
    WHERE N.TIPO_DOCUMENTO = 2 
      AND N.ESTADODOC = 3 
      AND N.FECHA >= '2026-01-01'
    """

    try:
        conn = pyodbc.connect(conn_str)
        cursor = conn.cursor()
        cursor.execute(query)
        
        rows = cursor.fetchall()
        print(f"📂 Se encontraron {len(rows)} registros en la BD. Iniciando extracción local...")

        for row in rows:
            # Procesamiento de datos binarios y metadatos
            blob_data = row[0]
            ext_raw = str(row[1]).strip() if row[1] else ".doc"
            extension = ext_raw if ext_raw.startswith('.') else f".{ext_raw}"
            
            # Construcción del nombre profesional del archivo
            nombre_archivo = f"{row[2]}-{row[3]}-{row[4]}-{row[5]}-{row[6]}-{row[7]}{extension}"
            filepath = os.path.join(output_dir, nombre_archivo)

            # Escritura del archivo físico
            if blob_data:
                with open(filepath, 'wb') as f:
                    f.write(blob_data)
                print(f"  ✅ Generado: {nombre_archivo}")

        conn.close()
        print("\n📂 Extracción de SQL Server completada con éxito.")

    except Exception as e:
        print(f"❌ Error durante la extracción: {e}")

# Ejecutamos la función
extraer_documentos_fiscalia()

📂 Se encontraron 19 registros en la BD. Iniciando extracción local...
  ✅ Generado: 6411-53972-2026-1-0-2.doc
  ✅ Generado: 6411-53981-2026-2-0-2.doc
  ✅ Generado: 6411-54017-2026-5-0-2.doc
  ✅ Generado: 6411-54052-2026-6-0-2.doc
  ✅ Generado: 6411-54146-2026-17-0-2.doc
  ✅ Generado: 6411-54281-2026-33-0-2.doc
  ✅ Generado: 6411-54320-2026-38-0-2.doc
  ✅ Generado: 6411-54356-2026-42-0-2.doc
  ✅ Generado: 6411-54362-2026-40-0-2.doc
  ✅ Generado: 6411-54368-2026-41-0-2.doc
  ✅ Generado: 6411-54533-2026-50-0-2.doc
  ✅ Generado: 6411-54106-2026-11-0-2.doc
  ✅ Generado: 6411-54129-2026-15-0-2.doc
  ✅ Generado: 6411-54517-2026-49-0-2.doc
  ✅ Generado: 6411-54221-2026-22-0-2.doc
  ✅ Generado: 6411-54311-2026-34-0-2.doc
  ✅ Generado: 6411-54438-2026-44-0-2.doc
  ✅ Generado: 6411-54458-2026-46-0-2.doc
  ✅ Generado: 6411-54478-2026-47-0-2.doc

📂 Extracción de SQL Server completada con éxito.


#Celda 3: Fase 2 - Configuración de Conexión a Azure Cloud
Esta celda demuestra la integración híbrida entre tu servidor local y la nube

In [7]:
# Recuperamos la cadena de conexión del .env
AZURE_CONN_STR = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
CONTAINER_NAME = "landing-zone"

# Inicializamos el cliente de Azure
try:
    blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
    print("✅ Conexión establecida con Azure Data Lake Gen2.")
except Exception as e:
    print(f"❌ Error de conexión: {e}")

✅ Conexión establecida con Azure Data Lake Gen2.


# Celda 4: Fase 3 - Ingestión Masiva a la "Landing Zone"
ingestión: subir los 19 archivos detectados en la carpeta local.

In [8]:
def subir_a_landing_zone(carpeta_local):
    archivos = [f for f in os.listdir(carpeta_local) if os.path.isfile(os.path.join(carpeta_local, f))]
    print(f"🚀 Iniciando carga masiva de {len(archivos)} archivos...")

    for nombre_archivo in archivos:
        ruta_completa = os.path.join(carpeta_local, nombre_archivo)
        blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=nombre_archivo)
        
        with open(ruta_completa, "rb") as data:
            blob_client.upload_blob(data, overwrite=True)
        print(f"  ✔ {nombre_archivo} -> Cargado.")

subir_a_landing_zone('../documentos')

🚀 Iniciando carga masiva de 19 archivos...
  ✔ 6411-53972-2026-1-0-2.doc -> Cargado.
  ✔ 6411-53981-2026-2-0-2.doc -> Cargado.
  ✔ 6411-54017-2026-5-0-2.doc -> Cargado.
  ✔ 6411-54052-2026-6-0-2.doc -> Cargado.
  ✔ 6411-54106-2026-11-0-2.doc -> Cargado.
  ✔ 6411-54129-2026-15-0-2.doc -> Cargado.
  ✔ 6411-54146-2026-17-0-2.doc -> Cargado.
  ✔ 6411-54221-2026-22-0-2.doc -> Cargado.
  ✔ 6411-54281-2026-33-0-2.doc -> Cargado.
  ✔ 6411-54311-2026-34-0-2.doc -> Cargado.
  ✔ 6411-54320-2026-38-0-2.doc -> Cargado.
  ✔ 6411-54356-2026-42-0-2.doc -> Cargado.
  ✔ 6411-54362-2026-40-0-2.doc -> Cargado.
  ✔ 6411-54368-2026-41-0-2.doc -> Cargado.
  ✔ 6411-54438-2026-44-0-2.doc -> Cargado.
  ✔ 6411-54458-2026-46-0-2.doc -> Cargado.
  ✔ 6411-54478-2026-47-0-2.doc -> Cargado.
  ✔ 6411-54517-2026-49-0-2.doc -> Cargado.
  ✔ 6411-54533-2026-50-0-2.doc -> Cargado.
